<a href="https://www.kaggle.com/code/viktorkondrashov123/nn-les2pract?scriptVersionId=293186182" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import torch
from torchvision import datasets, transforms

In [ ]:
data_dir1 = "/kaggle/input/leukemia-classification/C-NMC_Leukemia/training_data/fold_0"
data_dir2 = "/kaggle/input/leukemia-classification/C-NMC_Leukemia/training_data/fold_1"
data_dir3 = "/kaggle/input/leukemia-classification/C-NMC_Leukemia/training_data/fold_2"

In [ ]:
dataset1 = datasets.ImageFolder(data_dir1)
dataset2 = datasets.ImageFolder(data_dir2)
dataset3 = datasets.ImageFolder(data_dir3)

In [ ]:
datasets_all = torch.utils.data.ConcatDataset([dataset1, dataset2, dataset3])
data_train, data_test = torch.utils.data.random_split(datasets_all, [0.8, 0.2])

img, label = data_train[2000]

In [ ]:
train_transform = transforms.Compose(
    [transforms.Resize([64, 64]),
     transforms.CenterCrop(60),
     transforms.RandomRotation(180),
     transforms.ToTensor()
    ] 
)

test_transform = transforms.Compose(
    [transforms.Resize([64, 64]),
     transforms.CenterCrop(60),
     transforms.ToTensor()
    ] 
)

In [ ]:
transformed_img = train_transform(img)
print(transformed_img.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TransformDataset(Dataset):
    def __init__(self, dataset, transformer):
        self.dataset = dataset
        self.transformer = transformer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]

        img_new = self.transformer(img)

        return img_new, label

In [ ]:
train_data = TransformDataset(data_train, train_transform)

In [ ]:
test_data = TransformDataset(data_test, test_transform)

In [ ]:
train_loader = DataLoader(train_data, batch_size=256)
test_loader = DataLoader(test_data, batch_size=256)

In [ ]:
for img, idx in test_loader:
    print(img.shape)
    print(idx.shape)

In [ ]:
import matplotlib.pyplot as plt

for i in range(3):  # Show 3 images

    # Get the image data (tensor) and convert it back to a NumPy array for manipulation
    img, y = train_data[i]
    img = img.numpy()
    
    # Convert the color channels from (channels, height, width) to (height, width, channels) for pyplot
    img = img.transpose((1, 2, 0))
    print(img.shape)
    
    # Get the label name from the dataset class labels
    label = dataset1.classes[y]

    # Plot the image with a title (including label name)
    plt.imshow(img)
    plt.title(f"Label {label}")
    plt.show()

In [ ]:
from torchvision.utils import make_grid

loader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=32)

  
batch, labels = next(iter(loader))

grid = make_grid(batch).permute(1, 2, 0) # результатом є тензор

plt.imshow(grid)

In [ ]:
from torch import nn

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(10800, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)

device = "cuda"
model = model.to(device)

In [ ]:
# Функція втрат для класифікації
loss_fn = nn.CrossEntropyLoss()

# Оптимізатор (Adam) для оновлення ваг моделі
optimizer = torch.optim.Adam(
    model.parameters(),   # параметри нефромережі
    lr=0.2
)

loss_list = []
loss_test_list = []
for i in range(3):
    
    
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        result = model(imgs)
        loss = loss_fn(result, labels)
        print(f"loss: {loss}")

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loss_list.append(loss.cpu().item())

    # test data
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        result = model(imgs)
        loss = loss_fn(result, labels)
        print(f"loss: {loss}")

         
        loss_test_list.append(loss.cpu().item())

In [ ]:
img, label = train_data[57]
img = img.unsqueeze(0)
img.shape
img = img.to(device)
prediction = model(img)

print(label)
print(nn.Softmax()(prediction))



In [ ]:
import matplotlib.pyplot as plt

new_list = loss_list[20:]
plt.plot(new_list)

In [ ]:
plt.plot(new_test_list)